# Modélisation baseline

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
# Chargement des splits
df_train = pd.read_csv(r"..\data_finale\featuring\train_featured.csv", encoding="utf-8-sig")
df_val   = pd.read_csv(r"..\data_finale\featuring\val_featured.csv", encoding="utf-8-sig")

In [4]:
# Sélection de la cible (Y) et des 2 variables naïves (X)
X_cols_naives = ["Performance_Gls", "Playing Time_MP"]
target_col = "market_value_in_eur"

In [5]:
# Correction des NA
# On filtre les dataframes pour ne garder que les lignes qui ont toutes leurs données
# sur les variables prédictives (X) ET la cible (y).
cols_a_verifier = X_cols_naives + [target_col]

df_train_clean = df_train.dropna(subset=cols_a_verifier)
df_val_clean = df_val.dropna(subset=cols_a_verifier)

print(f"Lignes après suppression des NA -> Train: {len(df_train_clean)} (vs {len(df_train)}) | Val: {len(df_val_clean)} (vs {len(df_val)})\n")

Lignes après suppression des NA -> Train: 7404 (vs 7404) | Val: 2431 (vs 2431)



In [6]:
# On extrait du jeu d'entraînement uniquement les colonnes de performance choisies
X_train = df_train_clean[X_cols_naives]
# On extrait la colonne que le modèle doit apprendre à prédire (la valeur marchande réelle)
y_train = df_train_clean[target_col]

# On effectue exactement la même séparation sur le jeu de validation
X_val = df_val_clean[X_cols_naives]
y_val = df_val_clean[target_col]

print(f"Variables utilisées pour le modèle naïf : {X_cols_naives}")
print(f"Variable cible : {target_col}\n")

Variables utilisées pour le modèle naïf : ['Performance_Gls', 'Playing Time_MP']
Variable cible : market_value_in_eur



In [7]:
# Entraînement du modèle naïf (Régression Linéaire)
modele_naif = LinearRegression()
modele_naif.fit(X_train, y_train)

# Prédictions sur le jeu d'entraînement et de validation
y_pred_train = modele_naif.predict(X_train)
y_pred_val = modele_naif.predict(X_val)

In [8]:
# Calcul des métriques de performance
def calculer_metriques(y_reel, y_pred):
    mae = mean_absolute_error(y_reel, y_pred)
    rmse = np.sqrt(mean_squared_error(y_reel, y_pred))
    r2 = r2_score(y_reel, y_pred)
    return mae, rmse, r2

mae_train, rmse_train, r2_train = calculer_metriques(y_train, y_pred_train)
mae_val, rmse_val, r2_val = calculer_metriques(y_val, y_pred_val)

In [9]:
# Affichage des résultats
print("Performances du modèle naïf (baseline)")
print()
print(f"Jeu d'entraînement (train - saisons 2020-2023)")
print(f"- MAE  (Erreur Moyenne Absolue) : {mae_train:,.2f} €")
print(f"- RMSE (Écart-type des erreurs) : {rmse_train:,.2f} €")
print(f"- R²   (Pouvoir explicatif)     : {r2_train:.4f} ({r2_train*100:.1f}%)")
print()
print(f"Jeu de validation (val - saison 2024)")
print(f"- MAE  (Erreur Moyenne Absolue) : {mae_val:,.2f} €")
print(f"- RMSE (Écart-type des erreurs) : {rmse_val:,.2f} €")
print(f"- R²   (Pouvoir explicatif)     : {r2_val:.4f} ({r2_val*100:.1f}%)")

# Un petit aperçu visuel des erreurs
df_comparaison = pd.DataFrame({
    "Joueur": df_val_clean["player"],
    "Saison": df_val_clean["season_year"],
    "Valeur Réelle": y_val,
    "Prédiction Naïve": y_pred_val,
    "Erreur (Ecart)": np.abs(y_val - y_pred_val)
})

print("\nExemple de prédictions du modèle naïf sur le jeu de validation :")
display(df_comparaison.sort_values(by="Erreur (Ecart)", ascending=False).head(5))

Performances du modèle naïf (baseline)

Jeu d'entraînement (train - saisons 2020-2023)
- MAE  (Erreur Moyenne Absolue) : 8,404,524.93 €
- RMSE (Écart-type des erreurs) : 13,388,272.20 €
- R²   (Pouvoir explicatif)     : 0.2574 (25.7%)

Jeu de validation (val - saison 2024)
- MAE  (Erreur Moyenne Absolue) : 9,084,659.27 €
- RMSE (Écart-type des erreurs) : 15,493,254.01 €
- R²   (Pouvoir explicatif)     : 0.2589 (25.9%)

Exemple de prédictions du modèle naïf sur le jeu de validation :


,Joueur,Saison,Valeur Réelle,Prédiction Naïve,Erreur (Ecart)
2301,Vinicius Júnior,2023,180000000.0,3.539570e+07,1.446043e+08
1173,Jude Bellingham,2023,180000000.0,4.313904e+07,1.368610e+08
1288,Kylian Mbappé,2023,180000000.0,5.782046e+07,1.221795e+08
658,Erling Haaland,2023,180000000.0,5.835730e+07,1.216427e+08
706,Federico Valverde,2023,120000000.0,1.492721e+07,1.050728e+08


# Régression logistique

In [ ]:
N_BINS = 5

# Instanciation : 5 tranches basées sur les quantiles
discretizer = KBinsDiscretizer(
    n_bins=N_BINS, encode="ordinal", strategy="quantile", subsample=None
)

# Fit sur y_train et transform sur train + val
y_train_classe = discretizer.fit_transform(
    y_train.values.reshape(-1, 1)
).ravel()
y_val_classe = discretizer.transform(y_val.values.reshape(-1, 1)).ravel()

# Extraction des bornes trouvées par les quantiles
bin_edges = discretizer.bin_edges_[0]

# Ajustement pour couvrir de 0 à l'infini
bin_edges[0] = 0
bin_edges[-1] = np.inf

# Formatage des labels en Millions d'euros (M€)
labels_tranches = []
for i in range(len(bin_edges) - 1):
    low = bin_edges[i] / 1e6
    high = bin_edges[i + 1] / 1e6

    if i == 0:
        labels_tranches.append(f"<{high:.1f}M€")
    elif i == len(bin_edges) - 2:
        labels_tranches.append(f">{low:.1f}M€")
    else:
        labels_tranches.append(f"{low:.1f}-{high:.1f}M€")

print("Bornes calculées (Quantiles) :")
for idx, label in enumerate(labels_tranches):
    print(f"Classe {idx} : {label}")

Bornes calculées (Quantiles) :
Classe 0 : <1.5M€
Classe 1 : 1.5-3.5M€
Classe 2 : 3.5-7.0M€
Classe 3 : 7.0-17.0M€
Classe 4 : >17.0M€


c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [11]:
# Entraînement du modèle de régression logistique
modele_logit = LogisticRegression(
    max_iter=1000, 
    class_weight="balanced", 
    random_state=1308
)
modele_logit.fit(X_train, y_train_classe)

# Prédictions (renvoient directement des classes 0, 1, 2, 3, 4)
y_pred_train_classe = modele_logit.predict(X_train)
y_pred_val_classe   = modele_logit.predict(X_val)



print("\nRégression Logistique (Discrétisation K-Means)\n")
print(f"Accuracy train : {accuracy_score(y_train_classe, y_pred_train_classe):.4f}")
print(f"Accuracy val   : {accuracy_score(y_val_classe, y_pred_val_classe):.4f}\n")

print("Rapport de classification (validation) :")
print(classification_report(
    y_val_classe, 
    y_pred_val_classe, 
    target_names=labels_tranches
))

print("Matrice de confusion (validation) :")
cm = confusion_matrix(y_val_classe, y_pred_val_classe)
df_cm = pd.DataFrame(cm, index=labels_tranches, columns=labels_tranches)
print(df_cm)


Régression Logistique (Discrétisation K-Means)

Accuracy train : 0.3454
Accuracy val   : 0.3451

Rapport de classification (validation) :
              precision    recall  f1-score   support

      <1.5M€       0.38      0.71      0.49       380
   1.5-3.5M€       0.29      0.19      0.23       542
   3.5-7.0M€       0.22      0.19      0.20       416
  7.0-17.0M€       0.29      0.20      0.24       562
     >17.0M€       0.45      0.51      0.48       531

    accuracy                           0.35      2431
   macro avg       0.33      0.36      0.33      2431
weighted avg       0.33      0.35      0.32      2431

Matrice de confusion (validation) :
            <1.5M€  1.5-3.5M€  3.5-7.0M€  7.0-17.0M€  >17.0M€
<1.5M€         271         44         35          19       11
1.5-3.5M€      206        102         90          85       59
3.5-7.0M€       94         75         78          79       90
7.0-17.0M€     103         81         83         115      180
>17.0M€         46        

In [15]:
odds_ratios = np.exp(modele_logit.coef_)

# On met ça dans un DataFrame pandas
df_odds = pd.DataFrame(
    odds_ratios, columns=X_train.columns, index=modele_logit.classes_
)

# Affichage
print("Odds ratios par classe")
print(df_odds.T)

Odds ratios par classe
                      0.0       1.0       2.0       3.0       4.0
Performance_Gls  0.794850  0.938470  1.010146  1.101633  1.204684
Playing Time_MP  0.923146  0.984357  1.006060  1.029331  1.062669
